# 第六章 PyTorch 可视化模块（Visualization）

> 适合 Google Colab：边运行、边理解、后续快速复习。  
> 原教程顺序：**6.1 TensorBoard → 6.2 CNN 卷积核与特征图 → 6.3 混淆矩阵与训练曲线 → 6.4 CAM / Grad-CAM 与 Hook → 6.5 模型参数可视化**

## 学习目标

学完本章，你应能：

1. 用 TensorBoard 记录 loss、图像、参数分布和模型图；
2. 解释卷积核与特征图的 shape，并用 `make_grid` 可视化；
3. 从混淆矩阵读取类别偏好、Precision / Recall；
4. 通过训练/验证曲线识别欠拟合与过拟合迹象；
5. 理解 Grad-CAM 的“特征图 + 梯度权重”机制，并会用 hook 抓取中间结果；
6. 用 `torchinfo` / PyTorch 原生接口检查层级、shape、参数量和粗略内存。

## 本 Notebook 的取舍

原教程 6.2～6.4 使用 AlexNet / ResNet、CIFAR-10 与真实图片。本笔记改用**小型合成分类任务**，避免下载大型数据或预训练权重，同时保留同样的机制。

### 当前 API 修正

- `Conv2d.weight` 的准确 shape 是 `[out_channels, in_channels / groups, kH, kW]`。
- TorchVision 的 `pretrained=True/False` 已弃用；现代写法使用 `weights=...` 或 `weights=None`。
- 当前 `register_forward_hook` **可以通过返回值修改 output**；本章仅把它用于观测，不修改数据流。
- 旧 `register_backward_hook` 已弃用，使用 `register_full_backward_hook`。

教程入口：<https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-6/>

## 环境导入

只依赖 Colab 默认常见包：PyTorch、TorchVision、NumPy、Matplotlib。  
`tensorboard` 与 `torchinfo` 会在 Colab 中按需安装；在离线环境中缺失时会跳过对应外部工具演示，其余单元仍可运行。

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.utils import make_grid

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

IN_COLAB = importlib.util.find_spec("google.colab") is not None
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Running in Colab:", IN_COLAB)

# 6.1 TensorBoard 安装与使用

TensorBoard 的核心流程只有两步：

```text
训练代码 ──SummaryWriter──> event file ──TensorBoard──> Web UI
```

`SummaryWriter` 异步写日志，训练循环只需要持续记录数据。常用内容：

- `add_scalar`：loss / accuracy / learning rate；
- `add_histogram`：权重、梯度分布；
- `add_image(s)`：输入、特征图、卷积核；
- `add_figure`：Matplotlib 图；
- `add_graph`：模型计算图。

官方文档：<https://docs.pytorch.org/docs/stable/tensorboard>

In [ ]:
# Colab 通常已有 TensorBoard；如果缺失则按需安装。
if IN_COLAB and importlib.util.find_spec("tensorboard") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorboard"])

TB_AVAILABLE = importlib.util.find_spec("tensorboard") is not None
print("TensorBoard available:", TB_AVAILABLE)

## 6.1.1 `add_scalar`：训练中最重要的日志

建议使用分层 tag，例如 `Loss/train`、`Loss/valid`。这样 TensorBoard 会把同类指标组织在一起。

In [ ]:
log_dir = Path("runs/chapter6_demo")

if TB_AVAILABLE:
    from torch.utils.tensorboard import SummaryWriter
    writer = SummaryWriter(log_dir=str(log_dir))
    for step in range(10):
        writer.add_scalar("Loss/train", 1.0 / (step + 1), step)
        writer.add_scalar("Loss/valid", 1.2 / (step + 1) + 0.02, step)
    writer.close()
    print("event files:", [p.name for p in log_dir.glob("events.out.tfevents.*")])
else:
    print("当前环境未安装 tensorboard；Colab 中运行上一格会自动安装。")

## 6.1.2 图像、直方图与模型图

图像通常使用 `CHW`，批量图像使用 `NCHW`。`make_grid()` 可以先把 batch 拼成一张大图，再交给 `add_image()`。

In [ ]:
# 构造 8 张 1x16x16 小图，不下载数据集
sample_imgs = torch.rand(8, 1, 16, 16)
grid = make_grid(sample_imgs, nrow=4, normalize=True)
print("batch shape:", sample_imgs.shape)
print("grid shape :", grid.shape)

if TB_AVAILABLE:
    writer = SummaryWriter(log_dir=str(log_dir))
    writer.add_image("examples/random_images", grid, 0)
    writer.add_histogram("examples/random_values", torch.randn(1000), 0)

    graph_model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
    writer.add_graph(graph_model, torch.randn(1, 4))
    writer.close()

### 在 Colab 打开 TensorBoard

在 Colab 新建一个 Code Cell 执行：

```python
%load_ext tensorboard
%tensorboard --logdir runs/chapter6_demo
```

Notebook 主体不强制启动 Web UI，避免重复运行时产生多个 TensorBoard 进程。

# 6.2 CNN 卷积核与特征图可视化

原教程先介绍 `make_grid`，再分别观察卷积核和特征图。

## 6.2.1 `make_grid`

输入最常见是 `[B, C, H, W]`。它只负责**排版**，不会改变模型行为。

In [ ]:
imgs = torch.arange(6 * 1 * 8 * 8, dtype=torch.float32).reshape(6, 1, 8, 8)
grid = make_grid(imgs, nrow=3, normalize=True, scale_each=True)

print("input:", imgs.shape)
print("grid :", grid.shape)

plt.figure(figsize=(7, 3))
plt.imshow(grid.permute(1, 2, 0).squeeze())
plt.axis("off")
plt.title("make_grid result")
plt.show()

## 6.2.2 构造一个贯穿本章的 TinyCNN

我们生成一个简单二分类任务：亮条在左侧为类别 0，在右侧为类别 1。这样后续特征图、混淆矩阵和 Grad-CAM 都有明确含义。

In [ ]:
def make_synthetic_dataset(n=400, size=16, seed=42):
    g = torch.Generator().manual_seed(seed)
    x = 0.15 * torch.randn(n, 1, size, size, generator=g)
    y = torch.randint(0, 2, (n,), generator=g)

    for i, label in enumerate(y.tolist()):
        if label == 0:
            x[i, :, 3:13, 2:5] += 1.2   # 左侧亮条
        else:
            x[i, :, 3:13, 11:14] += 1.2 # 右侧亮条
    return x.clamp(0, 1), y

x_all, y_all = make_synthetic_dataset()
train_x, val_x = x_all[:320], x_all[320:]
train_y, val_y = y_all[:320], y_all[320:]

print("train:", train_x.shape, train_y.shape)
print("valid:", val_x.shape, val_y.shape)

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, 2),
        )

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        return self.head(x)

model = TinyCNN()
print(model)

## 6.2.3 卷积核可视化

`nn.Conv2d.weight` 的 shape：

$$
[C_{out},\; C_{in}/groups,\; k_H,\; k_W]
$$

对于 `conv1: 1 → 8`，因此权重是 `[8, 1, 3, 3]`：8 个输出通道，每个输出通道对应一个 3×3 卷积核。

In [ ]:
kernels = model.conv1.weight.detach().cpu()
print("conv1.weight shape:", kernels.shape)

kernel_grid = make_grid(kernels, nrow=4, normalize=True, scale_each=True)
plt.figure(figsize=(6, 3))
plt.imshow(kernel_grid.permute(1, 2, 0).squeeze())
plt.axis("off")
plt.title("conv1 kernels before training")
plt.show()

## 6.2.4 特征图可视化

特征图通常是 `[B, C, H, W]`。一个样本经过某卷积层后有 `C` 个通道，可以把每个通道当作一张灰度图。

教程先用“把层单独拿出来再 forward”的方式获得浅层特征图；更通用的方式是 6.4 的 hook。

In [ ]:
sample = train_x[:1]
with torch.no_grad():
    fmap1 = F.relu(model.conv1(sample))

print("input shape :", sample.shape)
print("feature map :", fmap1.shape)  # [1, 8, 16, 16]

fmap_grid = make_grid(fmap1[0].unsqueeze(1), nrow=4, normalize=True, scale_each=True)
plt.figure(figsize=(7, 4))
plt.imshow(fmap_grid.permute(1, 2, 0).squeeze())
plt.axis("off")
plt.title("conv1 feature maps")
plt.show()

# 6.3 混淆矩阵与训练曲线可视化

教程使用 CIFAR-10 + ResNet；这里用 TinyCNN 完成同一套分析。

先训练几轮，并同时保存 train / valid loss。训练曲线必须**成对观察**：只看训练 loss 无法判断泛化差距。

In [ ]:
torch.manual_seed(SEED)
model = TinyCNN()  # 重新初始化，保证实验可复现
optimizer = torch.optim.Adam(model.parameters(), lr=2e-2)

history = {"train_loss": [], "valid_loss": [], "valid_acc": []}

for epoch in range(8):
    model.train()
    optimizer.zero_grad()
    train_logits = model(train_x)
    train_loss = F.cross_entropy(train_logits, train_y)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        valid_logits = model(val_x)
        valid_loss = F.cross_entropy(valid_logits, val_y)
        valid_acc = (valid_logits.argmax(dim=1) == val_y).float().mean()

    history["train_loss"].append(train_loss.item())
    history["valid_loss"].append(valid_loss.item())
    history["valid_acc"].append(valid_acc.item())

print("final train loss:", history["train_loss"][-1])
print("final valid loss:", history["valid_loss"][-1])
print("final valid acc :", history["valid_acc"][-1])

## 6.3.1 训练曲线

典型判断：

- train / valid 都高：可能欠拟合（高偏差）；
- train 持续下降、valid 反而恶化：过拟合迹象（泛化差距增大）；
- 两者一起下降且差距小：通常是健康趋势。

这只是诊断信号，不应仅凭一张曲线下绝对结论。

In [ ]:
epochs = np.arange(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(6, 4))
plt.plot(epochs, history["train_loss"], marker="o", label="train")
plt.plot(epochs, history["valid_loss"], marker="o", label="valid")
plt.xlabel("epoch")
plt.ylabel("cross entropy loss")
plt.title("Training / validation loss")
plt.legend()
plt.show()

if TB_AVAILABLE:
    writer = SummaryWriter(log_dir=str(log_dir))
    for i, (tr, va) in enumerate(zip(history["train_loss"], history["valid_loss"])):
        writer.add_scalars("Loss_group", {"train_loss": tr, "valid_loss": va}, i)
    writer.close()

## 6.3.2 混淆矩阵（Confusion Matrix）

约定与教程一致：**行 = 真实类别，列 = 预测类别**。

对于第 $i$ 类：

$$
Recall_i = \frac{CM_{ii}}{\sum_j CM_{ij}},\qquad
Precision_i = \frac{CM_{ii}}{\sum_j CM_{ji}}
$$

原教程用 Python 循环逐样本计数；这里用 `torch.bincount` 向量化完成。

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(val_x).argmax(dim=1)

def confusion_matrix_torch(y_true, y_pred, num_classes):
    ids = y_true * num_classes + y_pred
    return torch.bincount(ids, minlength=num_classes**2).reshape(num_classes, num_classes)

cm = confusion_matrix_torch(val_y, y_pred, num_classes=2)
print(cm)
print("rows=true, cols=pred")

recall = cm.diag() / cm.sum(dim=1).clamp_min(1)
precision = cm.diag() / cm.sum(dim=0).clamp_min(1)
print("recall   :", recall.float())
print("precision:", precision.float())

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm.numpy())
ax.set_xticks([0, 1], labels=["left", "right"])
ax.set_yticks([0, 1], labels=["left", "right"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center")
plt.colorbar(im, ax=ax)
plt.show()

if TB_AVAILABLE:
    writer = SummaryWriter(log_dir=str(log_dir))
    writer.add_figure("validation/confusion_matrix", fig, global_step=7)
    writer.close()

> **教程勘误**：6.3 的猫狗示例矩阵写的是第二行为 `10, 20`，正文随后一处写成了 `8, 22`；Precision / Recall / Accuracy 的计算又按 `10, 20`。理解时应以给出的矩阵为准。

# 6.4 CAM 可视化与 Hook

CAM（Class Activation Mapping）的目标不是“看特征图长什么样”，而是回答：

> **模型为了某个类别，主要依赖输入的哪些空间区域？**

## 6.4.1 CAM → Grad-CAM

经典 CAM 依赖特定结构（最后卷积层 → GAP → 线性分类器）。Grad-CAM 更通用：用目标类别分数对特征图的梯度表示各通道的重要程度。

对于类别 $c$、第 $k$ 个特征图 $A^k$：

$$
\alpha_k^c = \frac{1}{Z}\sum_i\sum_j
\frac{\partial y^c}{\partial A_{ij}^k}
$$

$$
L_{Grad-CAM}^c = ReLU\left(\sum_k \alpha_k^c A^k\right)
$$

`ReLU` 保留对目标类别有正向贡献的区域。

## 6.4.2 Hook：不改 `forward()` 也能观察中间状态

本节最常用两类：

- `register_forward_hook`：拿到中间层输出（activation）；
- `register_full_backward_hook`：拿到该 Module 的反向梯度。

Hook 返回 `RemovableHandle`，用完应 `handle.remove()`，否则重复注册会导致重复执行。

当前 API 说明：forward hook 可以返回修改后的 output；本例只观测，不修改。

In [ ]:
# 先用一个最小例子观察 forward hook
captured = {}

def save_output(module, args, output):
    captured["shape"] = tuple(output.shape)

handle = model.conv1.register_forward_hook(save_output)
_ = model(val_x[:2])
handle.remove()

print("captured conv1 output shape:", captured["shape"])

## 6.4.3 手写最小 Grad-CAM

选择最后一个卷积层 `conv2` 作为 target layer：

1. forward hook 保存 feature maps；
2. backward hook 保存目标类别对该层输出的梯度；
3. 对梯度做空间平均，得到每个通道的权重；
4. 加权求和 + ReLU；
5. resize 回输入大小并归一化。

In [ ]:
def grad_cam(model, x, target_layer, class_idx=None):
    cache = {}

    def forward_hook(module, args, output):
        cache["activations"] = output.detach()

    def backward_hook(module, grad_input, grad_output):
        cache["gradients"] = grad_output[0].detach()

    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_full_backward_hook(backward_hook)

    model.eval()
    logits = model(x)
    if class_idx is None:
        class_idx = int(logits.argmax(dim=1).item())

    model.zero_grad(set_to_none=True)
    logits[0, class_idx].backward()

    activations = cache["activations"]       # [1, C, H, W]
    gradients = cache["gradients"]           # [1, C, H, W]
    weights = gradients.mean(dim=(2, 3), keepdim=True)

    cam = (weights * activations).sum(dim=1, keepdim=True)
    cam = F.relu(cam)
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)

    cam_min, cam_max = cam.amin(), cam.amax()
    cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)

    h1.remove()
    h2.remove()
    return cam.detach(), logits.detach(), class_idx

sample_idx = 0
x_one = val_x[sample_idx:sample_idx+1]
cam, logits, pred_class = grad_cam(model, x_one, model.conv2)

print("logits:", logits)
print("predicted class:", pred_class)
print("cam shape:", cam.shape)

In [ ]:
image = x_one[0, 0].numpy()
heatmap = cam[0, 0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(image)
axes[0].set_title("input")
axes[1].imshow(heatmap)
axes[1].set_title("Grad-CAM")
axes[2].imshow(image)
axes[2].imshow(heatmap, alpha=0.45)
axes[2].set_title("overlay")
for ax in axes:
    ax.axis("off")
plt.show()

### CAM 系列应该如何理解

- **CAM**：结构受限，需要 GAP + 线性分类头；
- **Grad-CAM**：用梯度估计通道重要性，适用范围更广；
- **Grad-CAM++**：进一步改进空间位置权重，尤其针对多个同类目标等情况。

热力图只是**解释工具**，不是“模型真正推理过程”的严格因果证明。高亮区域说明该解释方法认为这些区域对当前输出更重要。

# 6.5 模型参数可视化

原教程推荐 `torchinfo`，用于快速查看：

- Module 层级；
- 输入 / 输出 shape；
- 各层参数量；
- trainable / non-trainable 参数；
- 部分乘加量与内存估算。

相比已停止维护的 `torchsummary`，教程推荐 `torchinfo`。

In [ ]:
# torchinfo 不是 PyTorch 内置包；在 Colab 缺失时安装。
if IN_COLAB and importlib.util.find_spec("torchinfo") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchinfo"])

TORCHINFO_AVAILABLE = importlib.util.find_spec("torchinfo") is not None
print("torchinfo available:", TORCHINFO_AVAILABLE)

## 6.5.1 `torchinfo.summary`

对本章 TinyCNN，输入 shape 是 `[batch, 1, 16, 16]`。重点不是背输出格式，而是快速确认“每层 shape 是否符合预期、参数量是否合理”。

In [ ]:
if TORCHINFO_AVAILABLE:
    from torchinfo import summary
    stats = summary(
        model,
        input_size=(1, 1, 16, 16),
        col_names=("input_size", "output_size", "num_params", "mult_adds"),
        verbose=1,
    )
else:
    print("当前离线环境没有 torchinfo；Colab 中上一格会自动安装。")

## 6.5.2 不依赖第三方库：原生统计参数量与参数显存

参数数量：

$$
N_{params}=\sum_p p.numel()
$$

参数本身占用字节数：

$$
Memory=\sum_p p.numel()\times p.element\_size()
$$

但**训练显存远不止模型参数**：还包括梯度、optimizer state、activations、临时张量等。这个区别在 LLM 训练中非常重要。

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())

print(f"total params    : {total_params:,}")
print(f"trainable params: {trainable_params:,}")
print(f"parameter memory: {param_bytes / 1024**2:.4f} MiB")

for name, p in model.named_parameters():
    print(f"{name:20s} shape={tuple(p.shape)!s:16s} params={p.numel():5d}")

# 本章知识结构总结

```text
可视化
├─ 训练过程
│  ├─ TensorBoard scalar：loss / accuracy / lr
│  ├─ histogram：权重 / 梯度分布
│  └─ train + valid 曲线：拟合与泛化诊断
│
├─ CNN 内部
│  ├─ Conv2d.weight: [C_out, C_in/groups, kH, kW]
│  ├─ feature map: [B, C, H, W]
│  └─ make_grid：把多张二维图排成网格
│
├─ 分类结果
│  └─ Confusion Matrix：行=true，列=pred
│
├─ 模型解释
│  ├─ forward hook：抓 activation
│  ├─ backward hook：抓 gradient
│  └─ Grad-CAM：梯度均值作通道权重 → 加权特征图 → ReLU
│
└─ 模型规模
   ├─ torchinfo：层级 / shape / params / mult-adds
   └─ 原生 numel：参数量与参数内存
```

## 对 Transformer / LLM 最值得迁移的能力

CNN 的“卷积核 / 特征图”不会直接迁移到 Transformer，但本章真正通用的是：

1. 用 TensorBoard 监控 loss、learning rate、梯度分布；
2. 用 hook 抓取中间 activation / gradient；
3. 用 `named_parameters()`、参数量与显存估算理解模型规模；
4. 认识到“可视化相关性”不等于“证明因果解释”。

后续 Transformer 中，同类思路会用于观察 hidden states、attention、gradient norm、参数分布等。

# 学完必须会回答的问题

1. TensorBoard 为什么需要 `SummaryWriter` 和 event file？
2. `add_scalar`、`add_histogram`、`add_image` 分别适合记录什么？
3. `make_grid` 输入 `[B,C,H,W]` 时，第一维表示什么？
4. `Conv2d.weight` 四个维度的准确含义是什么？`groups` 为什么会改变第二维？
5. 卷积核（kernel）和特征图（feature map）有什么本质区别？
6. 混淆矩阵中“行=true、列=pred”时，Recall 和 Precision 分别应该按哪一维求和？
7. 为什么训练 loss 必须结合验证 loss 一起看？
8. forward hook 与 full backward hook 分别能获取什么？为什么用完要 `remove()`？
9. Grad-CAM 中为什么要对梯度在空间维度求平均？
10. Grad-CAM 最终为什么使用 ReLU？
11. `torchinfo` 能帮你快速发现哪些模型结构问题？
12. 为什么“参数大小”不能直接等同于“训练显存需求”？

## 参考

- PyTorch 实用教程（第二版）第六章：<https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-6/>
- TensorBoard：<https://docs.pytorch.org/docs/stable/tensorboard>
- `nn.Module` hooks：<https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html>
- `nn.Conv2d`：<https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html>
- TorchVision model weights API：<https://docs.pytorch.org/vision/stable/models.html>
- torchinfo：<https://github.com/TylerYep/torchinfo>

论文：
- Zhou et al., 2016, *Learning Deep Features for Discriminative Localization*, CVPR. arXiv:1512.04150.
- Selvaraju et al., 2017, *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization*, ICCV. arXiv:1610.02391.
- Chattopadhay et al., 2018, *Grad-CAM++: Generalized Gradient-Based Visual Explanations for Deep Convolutional Networks*, WACV. arXiv:1710.11063.